# Model 3 - XGBoost
**Owner:** _(your name)_  |  **Branch:** `model/xgboost`  |  **Stage 6 + 7** - model development, comparison and optimisation

Gradient-boosted trees: each new tree corrects the errors of the previous ones. Usually the strongest model on tabular data, with the most hyper-parameters to tune.

> **How to run:** first set `DATA_FRAC = 0.1` to check that everything works (a few minutes), then set it to `1.0` and *Run All* for the real results.
> Everything here uses the shared code in `src/` - same split, same 5 folds, same metrics as the other members.

## 1. Setup and data

In [ ]:
import sys; sys.path.append("../src")
import warnings; warnings.filterwarnings("ignore")
from common import *
from xgboost import XGBClassifier

NAME = "XGBoost"

# ---- run configuration ---------------------------------------------------------------------------
DATA_FRAC        = 1.0    # share of TRAINING patients used in this notebook (0.1 = quick check run)
EXP_FRAC         = 0.3    # share of patients used for the small experiments in section 4
TUNE_SAMPLE_FRAC = 0.3    # share of patients used DURING the search (the best model is refitted on ALL data)
N_ITER           = 15     # candidates tried by random / halving search
SEARCH_METHOD    = "random"   # "random" | "grid" | "halving"
# --------------------------------------------------------------------------------------------------
reset_log(NAME)

In [ ]:
X_train, X_test, y_train, y_test, g_train, cv, class_names = load_and_split()
X_train, y_train, g_train = patient_subsample(X_train, y_train, g_train, DATA_FRAC)
X_exp, y_exp, g_exp = patient_subsample(X_train, y_train, g_train, EXP_FRAC)
print("train", X_train.shape, "| test", X_test.shape, "| classes", class_names)
pd.Series(y_train).map(dict(enumerate(class_names))).value_counts(normalize=True).round(3)

## 2. Why this model? (algorithm selection and justification)

- **Task fit:** multiclass via softmax (`multi:softprob`), outputs calibrated-ish probabilities.
- **Data fit:** handles sparse one-hot input and missing values, captures non-linear interactions, built-in L1/L2 regularisation. **Weaknesses:** many hyper-parameters, can over-fit noisy data, less interpretable.
- **Imbalance:** multiclass XGBoost has no `class_weight`, so `BalancedXGB` (in `src/estimators.py`) applies balanced sample weights inside `fit()`.

## 3. Baseline model and validation strategy
- **Validation:** 5-fold `StratifiedGroupKFold` on the training set, grouped by `patient_nbr` (a patient never appears in both the training and validation part of a fold) and stratified (the rare `<30` class, about 11%, appears in every fold). The folds are fixed (`random_state=42`) and identical for every model.
- **Preprocessing inside the pipeline:** imputation, scaling and one-hot encoding are refitted on the training part of each fold, so nothing leaks from the validation part.
- **Imbalance:** handled with class weights, not by resampling (SMOTE works poorly on ~2,400 sparse one-hot columns).
- **Main metric: macro-F1** (every class counts equally). Accuracy is misleading here because always predicting `NO` already scores about 55%. We also track macro-recall, ROC-AUC (one-vs-rest) and PR-AUC / recall for the `<30` class.
- The **test set is not touched** until section 6.

In [ ]:
def make_pipe(**kw):
    params = dict(n_estimators=300, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", eval_metric="mlogloss", n_jobs=-1, random_state=RANDOM_STATE)
    params.update(kw)
    return Pipeline([("pre", make_preprocessor(scale=False)),
                     ("m", BalancedXGB(**params))])


pipe = make_pipe()
baseline = cv_evaluate(pipe, X_train, y_train, g_train, cv)
pd.Series({k: round(v, 4) for k, v in baseline.items() if not isinstance(v, list)})

### Baseline observations
_Write 2-3 sentences: How does the baseline compare with the majority-class score (macro-F1 about 0.23)? Is there a big gap between `train_macro_f1` and `cv_macro_f1` (over-fitting)?_

## 4. Experiments - understanding why the model behaves the way it does
Each experiment is cross-validated with the same folds on a patient sample (`EXP_FRAC`) and is saved automatically to `outputs/results/<model>_experiments.csv` (this is your experiment record for the report).

### Experiment 1
**Learning rate vs number of trees** - a small learning rate needs more trees but usually generalises better.

In [ ]:
for lr_, n in [(0.3, 100), (0.1, 300), (0.05, 600)]:
    run_experiment(NAME, "Learning rate x trees", f"lr={lr_}, trees={n}",
                   make_pipe(learning_rate=lr_, n_estimators=n, max_depth=4), X_exp, y_exp, g_exp, cv)
show_experiments(NAME, "Learning rate x trees")

**Observation:** _(what did you see and why? one or two sentences)_

### Experiment 2
**Tree depth** - controls how many feature interactions each tree can model.

In [ ]:
sweep(NAME, "max_depth", [3, 4, 6, 8], lambda v: make_pipe(max_depth=v, n_estimators=150), X_exp, y_exp, g_exp, cv)

**Observation:** _(what did you see and why? one or two sentences)_

### Experiment 3
**Feature engineering: do the 4 engineered features help?** (`total_previous_visits`, `has_previous_inpatient_visit`, `total_procedures`, `age_midpoint`)

In [ ]:
ENGINEERED = ["total_previous_visits", "has_previous_inpatient_visit", "total_procedures", "age_midpoint"]
run_experiment(NAME, "Engineered features", "with", make_pipe(max_depth=4, n_estimators=150), X_exp, y_exp, g_exp, cv)
run_experiment(NAME, "Engineered features", "without", make_pipe(max_depth=4, n_estimators=150),
               X_exp.drop(columns=ENGINEERED), y_exp, g_exp, cv)
show_experiments(NAME, "Engineered features")

**Observation:** _(what did you see and why? one or two sentences)_

### Experiment 4
**Imbalance handling** - balanced sample weights vs no weights. Compare macro-F1 with accuracy and `<30` recall (see the log file).

In [ ]:
plain = Pipeline([("pre", make_preprocessor(scale=False)),
                  ("m", XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                                      tree_method="hist", eval_metric="mlogloss", n_jobs=-1, random_state=RANDOM_STATE))])
run_experiment(NAME, "Imbalance", "balanced weights", make_pipe(max_depth=4, n_estimators=150), X_exp, y_exp, g_exp, cv)
run_experiment(NAME, "Imbalance", "no weights", plain, X_exp, y_exp, g_exp, cv)
show_experiments(NAME, "Imbalance")

**Observation:** _(what did you see and why? one or two sentences)_

## 5. Hyper-parameter tuning (Stage 7)
**Search strategy:** **Random search** (`RandomizedSearchCV`). Six hyper-parameters interact (learning rate x trees x depth x sampling), so a grid would explode; random search covers the space efficiently. If it is too slow, use `SEARCH_METHOD = "halving"`.

Scoring is **macro-F1** with the same grouped folds. To keep the search fast, it runs on `TUNE_SAMPLE_FRAC` of the patients and the best configuration is then **refitted on all training data**. The tuned model is re-evaluated with the full 5-fold CV so that baseline and tuned scores are directly comparable.

In [ ]:
space = {"m__max_depth": [3, 4, 6, 8],
         "m__learning_rate": [0.03, 0.05, 0.1, 0.2],
         "m__n_estimators": [200, 400, 600],
         "m__subsample": [0.7, 0.85, 1.0],
         "m__colsample_bytree": [0.6, 0.8, 1.0],
         "m__min_child_weight": [1, 3, 5]}


tune_res = tune(pipe, space, X_train, y_train, g_train, cv,
                method=SEARCH_METHOD, n_iter=N_ITER, sample_frac=TUNE_SAMPLE_FRAC)
print("Best parameters:", tune_res.best_params_)
print(f"Search took {tune_res.search_time_s/60:.1f} min")
top_configs(tune_res)

In [ ]:
tuned = cv_evaluate(tune_res.best_estimator_, X_train, y_train, g_train, cv)   # full 5-fold CV, comparable with baseline
compare_table(baseline, tuned)

**Observation:** _Did tuning help? By how much compared with the fold-to-fold spread (`cv_macro_f1_std`)? Which parameter mattered most?_

## 6. Final evaluation on the untouched test set (reported once)
The final model of the project is chosen later, in `compare_models.ipynb`, by **CV score - not by test score**.

In [ ]:
result = finalize(NAME, baseline, tuned, tune_res, X_test, y_test, class_names)

## 7. Interpretation

In [ ]:
best = tune_res.best_estimator_
tbl = feature_importance_table(best, X_train.columns)
top = top_original_features(tbl, 15)
plot_top_features(top, f"{NAME}: top 15 original features", NAME)
top.round(4)

## 8. Observations and viva preparation
**Write your own conclusions here (they go into the report):**
- Where does this model perform well / badly, per class? Why?
- What did the experiments show about over-fitting, imbalance and feature choices?
- Do the important features make clinical sense?

**Be ready to answer:**

- Boosting vs bagging: what does each new tree learn?
- What do `learning_rate`, `n_estimators`, `max_depth`, `subsample` and `colsample_bytree` control?
- Why do you need sample weights for imbalance in multiclass XGBoost?
- Did the engineered features help? Why might they add little for a tree model?
- How do you know the tuned model is not over-fitting?